<a href="https://colab.research.google.com/github/LizaHam123/sentiment-analysis-algerie-telecom/blob/main/Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch -q
!pip install transformers -q
!pip install tqdm pandas accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.3 MB/s eta 0:00:00


In [ ]:
# ANALYSE AMÉLIORÉE - VERSION CORRIGÉE
import torch
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import time

# Vérifier GPU
if not torch.cuda.is_available():
    print(" GPU NON DISPONIBLE!")
    exit()

print(f" GPU DISPONIBLE: {torch.cuda.get_device_name(0)}")

# Upload fichier
from google.colab import files
uploaded = files.upload()

# Charger données
df = pd.read_csv("Commentaires_traduits.csv", encoding='utf-8')
comment_column = df.columns[0]
comments = df[comment_column].dropna().tolist()
print(f" {len(comments)} commentaires à analyser")

# Configuration GPU optimisée
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
if gpu_memory >= 15:
    BATCH_SIZE = 64
elif gpu_memory >= 12:
    BATCH_SIZE = 32
else:
    BATCH_SIZE = 16

print(f"Batch size optimisé: {BATCH_SIZE}")

# Modèle CamemBERT
sentiment_classifier = pipeline(
    "text-classification",
    model="tblard/tf-allocine",
    device=0,
    batch_size=BATCH_SIZE,
    truncation=True,
    max_length=256,
    padding=True,
    return_all_scores=True
)

PROBLEM_KEYWORDS = {
    'fibre optique': [
        'fibre', 'fiber', 'optique', 'ftth', 'raccordement', 'raccorder',
        'raccordé', 'raccordement fibre', 'installation fibre', 'demande fibre',
        'éligibilité', 'éligible', 'couverture fibre', 'disponibilité fibre'
    ],
    'réseau': [
        'réseau', 'connexion', 'wifi', 'signal', 'coupure', 'déconnexion',
        'connecter', 'connecté', 'ligne', 'couverture', 'zone', 'antenne',
        'relais', 'infrastructure réseau', 'perte signal', 'mauvais signal'
    ],
    'service client': [
        'service', 'client', 'accueil', 'conseiller', 'support', 'personnel',
        'agent', 'représentant', 'réponse', 'contact', 'téléphone', 'call center',
        'assistance', 'aide', 'renseignement', 'information', 'réclamation'
    ],
    'facturation': [
        'facture', 'prix', 'tarif', 'coût', 'cher', 'abonnement', 'payer',
        'paiement', 'montant', 'frais', 'facturation', 'bonus', 'promotion',
        'offre', 'pack', 'forfait', 'recharge', 'crédit', 'solde', 'da', 'dinars'
    ],
    'débit internet': [
        'débit', 'vitesse', 'lent', 'internet', 'rapide', 'lenteur', 'megabit',
        'mbps', 'bande passante', 'téléchargement', 'upload', 'download',
        'streaming', 'navigation', 'connexion lente', 'vitesse internet'
    ],
    'matériel technique': [
        'technique', 'panne', 'installation', 'matériel', 'box', 'modem',
        'routeur', 'équipement', 'appareil', 'dispositif', 'configuration',
        'paramétrage', 'maintenance', 'réparation', 'défaillance', 'dysfonctionnement'
    ],
    'délais et attente': [
        'attente', 'délai', 'temps', 'rapidité', 'réactivité', 'patience',
        'mois', 'semaine', 'jour', 'retard', 'lenteur administrative',
        'traitement', 'suivi', 'dossier', 'demande en cours', 'en attente'
    ],
    'paiement électronique': [
        'paiement électronique', 'e-paiement', 'carte', 'ccp', 'virement',
        'versement', 'règlement', 'transaction', 'banque', 'compte',
        'paiement en ligne', 'digital', 'électronique'
    ],
    'qualité service': [
        'qualité', 'service', 'satisfaction', 'expérience', 'niveau',
        'performance', 'efficacité', 'professionnalisme', 'compétence',
        'amélioration', 'progrès', 'évolution'
    ]
}

print(" Configuration terminée avec mots-clés étendus!")

# Fonction améliorée avec meilleure détection neutre
def analyze_improved(comments_list):

    results = []
    print(f" Analyse améliorée de {len(comments_list)} commentaires...")
    for i in tqdm(range(0, len(comments_list), BATCH_SIZE), desc=" Analyse GPU"):
        batch = comments_list[i:i+BATCH_SIZE]
        try:
            sentiment_results = sentiment_classifier(batch)
            for j, sentiment_result in enumerate(sentiment_results):
                comment = batch[j]
                if isinstance(sentiment_result, list):
                    pos_score = next((s['score'] for s in sentiment_result if s['label'] == 'POSITIVE'), 0)
                    neg_score = next((s['score'] for s in sentiment_result if s['label'] == 'NEGATIVE'), 0)
                else:
                    if sentiment_result['label'] == 'POSITIVE':
                        pos_score = sentiment_result['score']
                        neg_score = 1 - pos_score
                    else:
                        neg_score = sentiment_result['score']
                        pos_score = 1 - neg_score
                score_difference = abs(pos_score - neg_score)
                if score_difference < 0.3:  # Scores très proches = neutre
                    sentiment = "neutre"
                elif pos_score > neg_score:
                    sentiment = "positif"
                else:
                    sentiment = "négatif"
                probleme = "N/A"
                if sentiment == "négatif":
                    comment_lower = comment.lower()
                    problem_scores = {}
                    for problem_type, keywords in PROBLEM_KEYWORDS.items():
                        score = 0
                        for keyword in keywords:
                            if keyword in comment_lower:
                                score += 1
                                if keyword in ['fibre', 'débit', 'facture', 'panne']:
                                    score += 1
                        if score > 0:
                            problem_scores[problem_type] = score
                    # Sélectionner le problème avec le score le plus élevéc
                    if problem_scores:
                        probleme = max(problem_scores, key=problem_scores.get)
                    else:
                        probleme = "autre"
                # Résultat final
                results.append({
                    "commentaire": comment,
                    "sentiment": sentiment,
                    "probleme": probleme
                })

        except Exception as e:
            print(f"Erreur batch: {e}")
            for comment in batch:
                results.append({
                    "commentaire": comment,
                    "sentiment": "erreur",
                    "probleme": "erreur_analyse"
                })

        # Sauvegarde
        if len(results) % 2000 == 0 and len(results) > 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(f"temp_ameliore_{len(results)}.csv", index=False, encoding='utf-8')
            print(f"\n Sauvegarde: {len(results)} commentaires")

    return results

# EXÉCUTION
print("\n" + "="*50)
print(" ANALYSE EN COURS")
print("="*50)

start_time = time.time()

# Analyse avec améliorations
results = analyze_improved(comments)

end_time = time.time()
duration = end_time - start_time

# Sauvegarde finale
df_final = pd.DataFrame(results)
df_final = df_final[["commentaire", "sentiment", "probleme"]]

filename = "resultats_sentiment.csv"
df_final.to_csv(filename, index=False, encoding='utf-8')

# Statistiques détaillées
print(f"\n ANALYSE TERMINÉE!")
print(f" Commentaires: {len(df_final)}")

# Répartition sentiments
sentiment_counts = df_final['sentiment'].value_counts()
total = len(df_final)

print(f"\n SENTIMENTS :")
for sentiment, count in sentiment_counts.items():
    percentage = (count/total)*100
    print(f"  {sentiment}: {count} ({percentage:.1f}%)")

# Répartition problèmes
negative_comments = df_final[df_final['sentiment'] == 'négatif']
if len(negative_comments) > 0:
    problem_counts = negative_comments['probleme'].value_counts()
    print(f"\n PROBLÈMES DÉTECTÉS :")
    for problem, count in problem_counts.items():
        percentage = (count/len(negative_comments))*100
        print(f"  {problem}: {count} ({percentage:.1f}%)")

# Affichage des 10 premiers commentaires
print(f"\nEXEMPLES DE RESULTAT:")
print("="*80)

for i in range(min(10, len(df_final))):
    row = df_final.iloc[i]
    commentaire = row['commentaire'][:100] + "..." if len(row['commentaire']) > 100 else row['commentaire']
    sentiment = row['sentiment']
    probleme = row['probleme']

    print(f"\n{i+1}. Commentaire: {commentaire}")
    print(f"   Sentiment: {sentiment.upper()}")

    if sentiment == "negatif" and probleme != "N/A":
        print(f"   Probleme: {probleme}")

    print("-" * 50)

# Télécharger
files.download(filename)


 GPU DISPONIBLE: Tesla T4


Saving Commentaires_traduits.csv to Commentaires_traduits (1).csv
 1325 commentaires à analyser
Batch size optimisé: 64


All model checkpoint layers were used when initializing TFCamembertForSequenceClassification.

All the layers of TFCamembertForSequenceClassification were initialized from the model checkpoint at tblard/tf-allocine.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFCamembertForSequenceClassification for predictions without further training.
Device set to use 0


 Configuration terminée avec mots-clés étendus!

 ANALYSE EN COURS
 Analyse améliorée de 1325 commentaires...


 Analyse GPU: 100%|██████████| 21/21 [04:50<00:00, 13.85s/it]


 ANALYSE TERMINÉE!
 Commentaires: 1325

 SENTIMENTS :
  négatif: 913 (68.9%)
  neutre: 305 (23.0%)
  positif: 107 (8.1%)

 PROBLÈMES DÉTECTÉS :
  fibre optique: 206 (22.6%)
  facturation: 136 (14.9%)
  débit internet: 130 (14.2%)
  réseau: 110 (12.0%)
  autre: 108 (11.8%)
  délais et attente: 80 (8.8%)
  service client: 77 (8.4%)
  matériel technique: 61 (6.7%)
  paiement électronique: 4 (0.4%)
  qualité service: 1 (0.1%)

EXEMPLES DE RESULTAT:

1. Commentaire: Comment peut-on en bénéficier ?;
   Sentiment: NEUTRE
--------------------------------------------------

2. Commentaire: L’ADSL est inexistante;
   Sentiment: NÉGATIF
--------------------------------------------------

3. Commentaire: Nous vous attendons, vous avez trop tardé haha;
   Sentiment: NÉGATIF
--------------------------------------------------

4. Commentaire: Incroyable mais vrai; ok je veux changer, comment faire s’il vous plaît ?;
   Sentiment: POSITIF
--------------------------------------------------

5. Comment

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>